<a href="https://colab.research.google.com/github/sruthi-kurra/dpo-vs-sft-qwen/blob/main/notebooks/02_Supervised_Fine_Tuning_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Setup

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes trl datasets

import json
import random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_DIR = Path("data")
OUTPUT_DIR = Path("checkpoints/sft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 2 — Load UltraFeedback Dataset & Create Deterministic Train/Eval Split

In [ ]:
from datasets import load_dataset

print("Loading UltraFeedback (train_sft)...")

dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_sft"
)

# Deterministic split (same every run)
dataset = dataset.shuffle(seed=SEED)

main = dataset.select(range(2500))

train_ds = main.select(range(2250))
eval_ds = main.select(range(2250, 2500))

print(f"Train examples: {len(train_ds)}")
print(f"Eval examples : {len(eval_ds)}")

## Cell 3 — Apply Qwen Chat Template & Prepare Training Dataset

In [ ]:
from transformers import AutoTokenizer

# Load tokenizer here directly so this cell never depends on Cell 4
# having been run first (survives runtime restarts / out-of-order execution)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_ds = train_ds.map(format_example)
eval_ds = eval_ds.map(format_example)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(train_ds[0]["text"][:500])

## Cell 4 — Load Qwen2.5-0.5B-Instruct with 4-bit QLoRA

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# (tokenizer already loaded in Cell 3 — no need to reload here)

# ------------------------------------------------------------
# 4-bit quantization config
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ------------------------------------------------------------
# Load base model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# ------------------------------------------------------------
# LoRA configuration
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

model.print_trainable_parameters()

print("\nModel loaded successfully.")
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")

## Cell 5 — Prepare Dataset for SFTTrainer

In [ ]:
train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c != "text"]
)

eval_ds = eval_ds.remove_columns(
    [c for c in eval_ds.column_names if c != "text"]
)

print(train_ds.column_names)
print(eval_ds.column_names)

## Cell 6 — Configure SFT Training

In [ ]:
from trl import SFTConfig

total_train_examples = len(train_ds)
effective_batch_size = 4 * 4  # per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = total_train_examples // effective_batch_size
warmup_steps = max(1, int(0.03 * steps_per_epoch))  # same 3% warmup, just pre-computed

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,   # replaces warmup_ratio
    weight_decay=0.01,
    max_grad_norm=0.3,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    bf16=torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,

    max_length=1024,
    dataset_text_field="text",
    packing=False,
)

print("SFT training config ready.")
print(f"Steps per epoch: {steps_per_epoch}, warmup_steps: {warmup_steps}")
print(f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"Output dir: {sft_config.output_dir}")

## Cell 7 — Train SFT Model

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,   # renamed from `tokenizer=` in trl>=0.16
)

print("Starting SFT training...")
train_result = trainer.train()

print("\nTraining complete.")
print(f"Final train loss: {train_result.training_loss:.4f}")

## Cell 8 — Evaluate & Save SFT Model

In [ ]:
# ------------------------------------------------------------
# Cell 8 — Evaluate & Save SFT Model
# ------------------------------------------------------------

# Final eval on held-out set
eval_metrics = trainer.evaluate()
print("Final eval metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# ------------------------------------------------------------
# Save LoRA adapter + tokenizer
# ------------------------------------------------------------

SFT_ADAPTER_DIR = OUTPUT_DIR / "sft_adapter_final"
SFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SFT_ADAPTER_DIR))
tokenizer.save_pretrained(str(SFT_ADAPTER_DIR))

print(f"\nSFT LoRA adapter saved to: {SFT_ADAPTER_DIR}")
print("Contents:", [p.name for p in SFT_ADAPTER_DIR.iterdir()])

# ------------------------------------------------------------
# Quick sanity check: generate a sample response
# ------------------------------------------------------------

model.eval()
sample_prompt = eval_ds[0]["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"
inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("\nSample generation from fine-tuned SFT model:")
print(generated)

## Cell 9 — Free Memory & Load DPO Preference Dataset

In [ ]:
# ------------------------------------------------------------
# Cell 9 — Free SFT Memory & Prepare DPO Preference Dataset
# ------------------------------------------------------------

import gc

# Free the SFT training model/trainer from GPU memory before DPO
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

# ------------------------------------------------------------
# Load UltraFeedback with chosen/rejected pairs for DPO
# ------------------------------------------------------------

from datasets import load_dataset

print("Loading UltraFeedback (train_prefs / test_prefs)...")

dpo_dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

# Same deterministic split pattern as Cell 2
dpo_dataset = dpo_dataset.shuffle(seed=SEED)

dpo_main = dpo_dataset.select(range(2500))
dpo_train_ds = dpo_main.select(range(2250))
dpo_eval_ds = dpo_main.select(range(2250, 2500))

print(f"DPO train examples: {len(dpo_train_ds)}")
print(f"DPO eval examples: {len(dpo_eval_ds)}")
print(f"\nColumns available: {dpo_dataset.column_names}")
print(f"\nSample chosen: {dpo_train_ds[0]['chosen']}")
print(f"\nSample rejected: {dpo_train_ds[0]['rejected']}")

In [ ]:
!zip -r sft_adapter_final.zip checkpoints/sft/sft_adapter_final

In [ ]:
from google.colab import files
files.download("sft_adapter_final.zip")

In [ ]:
import json
from pathlib import Path

Path("results").mkdir(exist_ok=True)

sft_metrics = {
    "eval_loss": 1.5751,
    "eval_entropy": 1.5702,
    "eval_num_tokens": 1016126,
    "eval_mean_token_accuracy": 0.6382,
    "final_train_loss": 1.6011,
    "epochs": 1,
    "steps": 141,
    "effective_batch_size": 16
}

with open("results/sft_metrics.json", "w") as f:
    json.dump(sft_metrics, f, indent=2)

print("Saved results/sft_metrics.json")

In [ ]:
from google.colab import files
files.download("results/sft_metrics.json")

In [ ]:
!pip freeze | grep -E "^(transformers|trl|peft|bitsandbytes|datasets|torch|accelerate)==" > requirements.txt

In [ ]:
from google.colab import files
files.download("requirements.txt")